# RHO-FFT · P0（Colab）：黄金值、基线回归与 E0 复现

这个 notebook 在 Colab 的 GPU 上执行里程碑 P0 里需要 GPU 的步骤，最后生成一份反馈报告。全部逻辑在仓库的 `experiments/p0/p0.py` 里，notebook 只负责搭环境、按顺序调用、显示结果。

**运行前**

1. 运行时类型选 GPU：默认用 G4，模型较大时可以换 H100。黄金值、回归结果和计时都按 GPU 型号分开保存，换型号不会混用。
2. 代码来源二选一（在下面的参数里设 `REPO_SOURCE`）：
   - `github`：先把分支 `p0-infra` 推到 GitHub。仓库是私有的，请在 Colab 左侧的"密钥"里添加 `GITHUB_TOKEN`（对该仓库有只读权限的 token），并允许本 notebook 访问。没有设置密钥时，notebook 会提示输入。
   - `bundle`：在本地仓库里执行 `git bundle create rho-fft.bundle --all`，把生成的文件放到 Google Drive 的 `rho-fft/` 目录。这种方式不需要推送。
3. 从上到下运行全部单元格。

**步骤一览**

| 步骤 | 内容 | 用不用 GPU |
|---|---|---|
| A1 到 A3 | 会话与 Drive、取代码、建环境 | 不用 |
| B | 预演：小模型把整条流程走一遍，约两分钟 | 不用 |
| C1 | 环境记录与单元测试 | 探测一次 |
| C2 | 黄金值与基线回归（I1、I2） | 用 |
| C3 | E0 复现，按时间预算排队 | 用 |
| D | 反馈报告与曲线 | 不用 |
| E | 各步骤状态汇总，10 秒后释放 GPU | |

每个代码单元格都有标题，可以折叠。B 或 C1 失败时 notebook 会停下；开启 `RELEASE_GPU` 时停下前同样会释放 GPU，避免空占。

**时间预算**

`BUDGET_HOURS` 是整个 notebook 的总预算，从第 1 步开始计时。E0 阶段在每种配置第一次运行前先跑 3 轮预检并计时，再按估计耗时决定是否启动；估计放不进剩余预算的运行会被跳过，下次运行本 notebook 时自动接着跑（开启 `USE_DRIVE` 时，已完成的运行保存在 Drive 上）。默认计划 `qwen` 只跑 Qwen2.5-1.5B / AG News 种子 42 的 4 次运行（HO-FL β=0.75 和 β=0、FedAvg、DeComFL）；`full` 按 OPT 种子 42、Qwen 种子 42、其余种子的顺序排队，预算内能跑多少跑多少。

黄金值、回归结果和计时都按 GPU 型号分开保存；换了 GPU 型号的会话会重新记录，不会拿另一种 GPU 的结果来比较。

**结束后带回什么**

最后一步显示的 `feedback_P0.md`，或者同目录下的 `feedback_P0_bundle.zip`。把它交回代码库这边，就能填写 `docs/reports/P0.md` 并决定下一轮改什么。

In [ ]:
#@title Parameters { display-mode: "form" }
# Where the code comes from.
REPO_SOURCE = "github"  #@param ["github", "bundle"]
REPO_URL = "https://github.com/GuangLun2000/RHO-FFT.git"  #@param {type:"string"}
REPO_REF = "p0-infra"  #@param {type:"string"}
BUNDLE_PATH = "/content/drive/MyDrive/rho-fft/rho-fft.bundle"  #@param {type:"string"}
ASK_FOR_TOKEN = True  #@param {type:"boolean"}

# Keep finished runs on Google Drive so that a later session can continue.
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/rho-fft"  #@param {type:"string"}

# Total wall-clock budget of this notebook, in hours.
BUDGET_HOURS = 5.5  #@param {type:"number"}

# "venv": Python 3.11 with the pinned dependencies of pyproject.toml (recommended).
# "system": Colab's own Python and torch; only a fallback, versions then differ from the pins.
ENV_MODE = "venv"  #@param ["venv", "system"]

# Stages.
RUN_TINY_DRY_RUN = True  #@param {type:"boolean"}
RUN_TESTS = True  #@param {type:"boolean"}
RUN_GOLDEN_AND_REGRESSION = True  #@param {type:"boolean"}
RUN_E0 = True  #@param {type:"boolean"}
# The queue is ordered by priority and stops at the budget: OPT seed 42 (5 runs), Qwen seed 42 (4 runs), then the
# remaining seeds of OPT and of Qwen. "full" takes as many of the 37 runs as fit; "lite" stops after seed 42;
# "opt" runs OPT seed 42 only; "qwen" runs Qwen2.5-1.5B on AG News, seed 42 only (HO-FL beta 0.75 and 0, FedAvg,
# DeComFL), which is the main setting of the paper.
E0_PLAN = "qwen"  #@param ["qwen", "opt", "lite", "full"]
# Run E0 even when the regression check did not pass (not recommended).
FORCE_E0 = False  #@param {type:"boolean"}
DOWNLOAD_BUNDLE = False  #@param {type:"boolean"}
# Release the GPU runtime 10 seconds after the last cell (and when the notebook stops on a failed check).
# Results are on Drive when USE_DRIVE is set; without Drive, download the bundle before the runtime goes away.
RELEASE_GPU = True  #@param {type:"boolean"}

## A1. 会话、GPU 与 Drive

记录开始时间（时间预算从这里算起），显示 GPU，按需挂载 Drive。

In [ ]:
#@title A1 · Session, GPU, Drive
import base64
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

SESSION_START = time.time()
DEADLINE = SESSION_START + 3600 * BUDGET_HOURS
REPO_DIR = "/content/RHO-FFT"
WORK = "/content/work/p0"            # results of this notebook (local disk, fast)
TINY_WORK = "/content/work/p0-tiny"  # results of the CPU dry run, kept apart from the real ones
CACHE = "/content/cache"             # models, datasets, token caches and manifests
VENV = "/content/venv"


def sh(command, cwd=None, env=None, check=True):
    # Run a command, stream its output into the cell, and stop on failure when check is set.
    shown = command if isinstance(command, str) else " ".join(str(part) for part in command)
    print(f"$ {shown}", flush=True)
    process = subprocess.Popen(command, shell=isinstance(command, str), cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {shown}")
    return code


def hours_left():
    return (DEADLINE - time.time()) / 3600


STATUS = {}  # one line per stage, printed in the last cell


def release_gpu(delay=10):
    # Hand the GPU runtime back. Everything on the VM's local disk is lost, so this runs only after the results
    # have been mirrored or shown.
    if not RELEASE_GPU:
        print("RELEASE_GPU is off; the runtime stays connected.")
        return
    print(f"Releasing the GPU runtime in {delay} seconds ...", flush=True)
    time.sleep(delay)
    from google.colab import runtime
    runtime.unassign()


def stop(message):
    # A failed check: release the GPU instead of leaving it idle, then stop "Run all".
    print(message, flush=True)
    release_gpu()
    raise RuntimeError(message)


sh("nvidia-smi", check=False)
MIRROR = ""
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    MIRROR = f"{DRIVE_DIR}/p0"
    os.makedirs(MIRROR, exist_ok=True)
    print("finished runs are mirrored to", MIRROR)

## A2. 取代码

克隆仓库并检出 `REPO_REF`。流程要用 `git archive` 取出两个参照提交（上游原样导入、插桩提交），所以需要带完整历史的克隆。token 只通过环境变量传给 git，不会出现在输出里。

In [ ]:
#@title A2 · Get the code
def git_environment():
    # GitHub credentials for a private repository, passed to git through the environment only.
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as error:
        print("Colab secret GITHUB_TOKEN not available:", type(error).__name__)
    if not token and ASK_FOR_TOKEN:
        from getpass import getpass
        token = getpass("GitHub token with read access (leave empty for a public repository): ").strip()
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    return env


if REPO_SOURCE == "bundle":
    if not os.path.exists(BUNDLE_PATH):
        raise FileNotFoundError(f"{BUNDLE_PATH} not found. Create it with `git bundle create rho-fft.bundle --all` and upload it.")
    source, git_env = BUNDLE_PATH, None
else:
    source, git_env = REPO_URL, git_environment()

if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh(["git", "clone", "--quiet", source, REPO_DIR], env=git_env)
else:
    sh(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin"], env=git_env)
remote_ref = f"origin/{REPO_REF}"
has_remote_ref = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", remote_ref],
                                capture_output=True).returncode == 0
sh(["git", "-C", REPO_DIR, "checkout", "--quiet", "--detach", remote_ref if has_remote_ref else REPO_REF])
sh(["git", "-C", REPO_DIR, "log", "--oneline", "-8"])

revisions = json.loads(Path(f"{REPO_DIR}/experiments/p0/revisions.json").read_text())
for name, sha in revisions.items():
    if subprocess.run(["git", "-C", REPO_DIR, "cat-file", "-e", sha + "^{commit}"], capture_output=True).returncode != 0:
        raise RuntimeError(f"Reference revision {name} ({sha}) is missing from this clone; a full-history clone is required.")
print("reference revisions present:", revisions)

## A3. 环境

默认建一个独立的 Python 3.11 虚拟环境，按 `pyproject.toml` 的固定版本安装依赖（PyTorch 2.6.0 / CUDA 12.4，与上游 README 一致）。所有训练都在这个环境的子进程里运行，不影响 Colab 自带的 Python。仓库本身不安装，运行时通过 `PYTHONPATH` 指向对应的源码树。

In [ ]:
#@title A3 · Environment
def clean_environment():
    # Colab exports constraint files for its own package set (PIP_CONSTRAINT, UV_CONSTRAINT, ...). They must not
    # apply to the separate environment built here, which pins different versions.
    return {key: value for key, value in os.environ.items() if not key.startswith(("UV_", "PIP_"))}


if ENV_MODE == "venv":
    uv = f"{sys.executable} -m uv"
    sh(f"{sys.executable} -m pip install -q uv")
    if not os.path.exists(f"{VENV}/bin/python"):
        if sh(f"{uv} venv --python 3.11 {VENV}", env=clean_environment(), check=False) != 0:
            print("Python 3.11 could not be provisioned; building the environment on the runtime's own Python instead")
            shutil.rmtree(VENV, ignore_errors=True)
            sh(f"{uv} venv --python {sys.executable} {VENV}", env=clean_environment())
    install = f"{uv} pip install --python {VENV}/bin/python -r pyproject.toml --extra dev"
    if sh(install + " --torch-backend cu124", cwd=REPO_DIR, env=clean_environment(), check=False) != 0:
        print("retrying without --torch-backend (the default Linux wheel of torch 2.6.0 is the CUDA 12.4 build)")
        sh(install, cwd=REPO_DIR, env=clean_environment())
    PY = f"{VENV}/bin/python"
else:
    PY = sys.executable
    sh(f"{PY} -m pip install -q transformers==4.56.2 peft==0.17.1 datasets==3.6.0 accelerate==1.10.1 "
       "safetensors==0.6.2 sentencepiece==0.2.1 pytest", env=clean_environment())

sh([PY, "-c", "import torch, transformers, peft, datasets; "
              "print('torch', torch.__version__, '| cuda runtime', torch.version.cuda, '| cuda available', torch.cuda.is_available()); "
              "print('transformers', transformers.__version__, '| peft', peft.__version__, '| datasets', datasets.__version__)"])


def p0(*stages, root=WORK, extra=()):
    # Run stages of experiments/p0/p0.py with the training interpreter; returns the exit code (0 = all stages ok).
    command = [PY, "experiments/p0/p0.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda", *extra]
    if MIRROR and root == WORK:
        command += ["--mirror", MIRROR]
    return sh(command, cwd=REPO_DIR, check=False)

## B. 预演（CPU，约两分钟）

用随机初始化的小模型和合成数据把整条流程走一遍：导出两个参照提交、跑冒烟配置、比较、E0 排队、生成反馈。它只检查流程能不能跑通，数字没有意义。这一步失败时 notebook 会停下，不消耗 GPU 时间；请把输出带回。

In [ ]:
#@title B · Dry run on CPU
if RUN_TINY_DRY_RUN:
    code = sh([PY, "experiments/p0/p0.py", "env", "golden", "regress", "e0", "feedback", "--tiny", "--quiet",
               "--root", TINY_WORK], cwd=REPO_DIR, check=False)
    STATUS["dry run"] = "ok" if code == 0 else "FAILED"
    if code != 0:
        stop("The dry run failed. Nothing has been run on the GPU yet; send the output above back.")
    print(f"\ndry run ok, {hours_left():.2f} h of budget left")

## C1. 环境记录与单元测试

`env` 记录 GPU 型号、驱动和各依赖的版本（黄金值与硬件有关，需要一起记录）。`tests` 在固定版本的环境里跑一遍不需要 GPU 的测试。

In [ ]:
#@title C1 · Environment record and unit tests
env_ok = p0("env") == 0
STATUS["environment"] = "ok" if env_ok else "FAILED"
if not env_ok:
    stop("Environment check failed (no CUDA device, bfloat16 not usable, or a reference revision is missing). See the warnings above.")
if RUN_TESTS:
    tests_ok = p0("tests") == 0
    STATUS["unit tests"] = "ok" if tests_ok else "FAILURES"
    print("unit tests:", "ok" if tests_ok else "FAILURES (listed above and in the feedback report)")

## C2. 黄金值与基线回归（I1、I2）

- `golden`：三个冒烟配置（HO-FL、FedAvg、DeComFL，OPT-125M / SST-2，3 轮）在**上游原样提交**上各跑一次，在**插桩提交**上各跑两次，再跑一次 `ho-fl-replay`。两次运行逐位相同就按逐位比较；否则比较最终参数，容差为最大绝对误差 1e-7（`replay.py` 的绝对容差，不含它另外允许的 1e-5 相对容差），同时要求每轮选中的客户端、起点和日志行数完全相同。
- `regress`：同样的配置在**当前代码**上跑一次，与黄金值比较逐轮 `task_loss`、评估指标和最终参数哈希，并在当前代码上跑 `ho-fl-replay`。

读法：表里 `I1 verdict` 为 `PASS_BITWISE` 或 `PASS_TOLERANCE` 即通过；`FAIL` 说明重排改变了基线，`first difference` 一列给出最早出现差异的位置；`INCONCLUSIVE` 说明这块 GPU 上参照提交自己两次运行的差就超过了容差，需要研究负责人决定怎么判。三棵源码树各用自己的缓存目录，所以数据划分和分词也在比较范围内。

In [ ]:
#@title C2 · Golden values and regression (I1, I2)
regression_ok = None
if RUN_GOLDEN_AND_REGRESSION:
    golden_ok = p0("golden") == 0
    regression_ok = golden_ok and p0("regress") == 0
    STATUS["golden values"] = "ok" if golden_ok else "ATTENTION"
    STATUS["regression (I1, I2)"] = "ok" if regression_ok else "ATTENTION"
    print("\ngolden:", "ok" if golden_ok else "ATTENTION", "| regression (I1, I2):", "ok" if regression_ok else "ATTENTION")
    print(f"{hours_left():.2f} h of budget left")

## C3. E0 复现（按时间预算）

每种配置在第一次运行前先跑 3 轮预检：既是冒烟检查（下载、显存、代码路径），也用来测每轮和每次评估的耗时。队列顺序是 OPT（种子 42）、Qwen（种子 42）、其余种子，所以便宜的配置先出结果。估计放不进剩余预算的运行会标为 `SKIPPED_BUDGET`，预检失败的配置标为 `SKIPPED_PREFLIGHT` 并附原因。

运行过程中每 10 轮打印一次训练损失，每次评估打印准确率。回归检查没有通过时默认不跑 E0，因为那样的结果不能当作基线。

In [ ]:
#@title C3 · E0 within the time budget
if RUN_E0:
    if regression_ok is False and not FORCE_E0:
        print("E0 skipped: the regression check did not pass. Set FORCE_E0 = True to run it anyway.")
    elif hours_left() < 0.2:
        print("E0 skipped: less than 12 minutes of budget left.")
    else:
        if regression_ok is None:
            print("NOTE: the regression check was not run in this session; E0 results are only a baseline if it passed before.")
        e0_done = p0("e0", extra=["--plan", E0_PLAN, "--deadline", str(DEADLINE)]) == 0
        STATUS["E0"] = "all planned runs completed" if e0_done else "partly done (skipped or failed runs are listed in the report)"
        print(f"{hours_left():.2f} h of budget left")

## D. 反馈报告

把上面所有结果汇总成一份报告：G0 检查表、环境、测试、黄金值与可复现性、I1 与 I2、E0 结果表（含上行字节数和产生结果的代码版本）、HO-FL 运行的逐段一阶供给（从 `ledger.jsonl` 统计，可与说明附录 A 的数值对照）、每次运行的曲线、耗时与 GPU 小时数、运行清单（含失败的运行和日志末尾）、结论与下一步。

In [ ]:
#@title D · Feedback report and curves
from IPython.display import Markdown, display

p0("feedback", extra=["--quiet"])
feedback_dir = Path(WORK) / "feedback"
display(Markdown((feedback_dir / "feedback_P0.md").read_text()))
STATUS["feedback"] = "written"

# Accuracy curves of the completed E0 runs, one panel per model and dataset.
data = json.loads((feedback_dir / "feedback_P0.json").read_text())
runs = data["e0"]["runs"]
if runs:
    import matplotlib.pyplot as plt

    panels = sorted({(run["group"][0], run["group"][1]) for run in runs})
    figure, axes = plt.subplots(1, len(panels), figsize=(6.5 * len(panels), 4.2), squeeze=False)
    for axis, panel in zip(axes[0], panels):
        for run in runs:
            if (run["group"][0], run["group"][1]) != panel:
                continue
            method, beta = run["group"][3], run["group"][4]
            label = (f"hofl beta={beta:g}" if method == "hofl" else method) + f" seed {run['seed']}"
            axis.plot([p["round"] for p in run["curve"]], [p["accuracy"] for p in run["curve"]], marker="o", markersize=3, label=label)
        axis.set_title(f"{panel[0]} / {panel[1]}")
        axis.set_xlabel("round")
        axis.set_ylabel("accuracy (%)")
        axis.grid(alpha=0.3)
        axis.legend(fontsize=8)
    figure.tight_layout()
    figure.savefig(feedback_dir / "e0_curves.png", dpi=150)
    plt.show()
else:
    print("no completed E0 run yet")

if MIRROR:
    shutil.copytree(feedback_dir, Path(MIRROR) / "feedback", dirs_exist_ok=True)
    print("feedback copied to", Path(MIRROR) / "feedback")
print("report:", feedback_dir / "feedback_P0.md")
print("bundle:", feedback_dir / "feedback_P0_bundle.zip")
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h")
if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(feedback_dir / "feedback_P0_bundle.zip"))

## E. 汇总并释放 GPU

打印各步骤的状态，10 秒后释放 GPU 运行时。释放后虚拟机本地磁盘上的内容会消失：开启 `USE_DRIVE` 时结果已经在 Drive 上；没开 Drive 时请先把 `DOWNLOAD_BUNDLE` 打开，或把 `RELEASE_GPU` 关掉。

In [ ]:
#@title E · Summary, then release the GPU
checklist = data.get("checklist", {})
print("stage status")
for name, state in STATUS.items():
    print(f"  {name:22s} {state}")
print("gate G0 checklist:", ", ".join(f"{key}={value}" for key, value in checklist.items()))
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h of {BUDGET_HOURS} h")
if MIRROR:
    print("results on Drive:", MIRROR)
elif RELEASE_GPU:
    print("WARNING: USE_DRIVE is off, so the results exist only in this runtime and in the output above.")
release_gpu(10)

## 带回代码库的内容

- **`feedback_P0.md`**：上面显示的报告。第 1 节是 G0 检查表，第 9 节是结论和下一步。
- **`feedback_P0_bundle.zip`**：报告、环境记录、黄金值、各次运行的 `config.json`、`summary.json`、`rounds.jsonl`、`curves.jsonl`、计时和日志。排查问题时用它。
- 黄金值文件 `golden/<提交>-<GPU>/golden.json` 会随 bundle 一起带回，之后提交到仓库的 `tests/golden/`。

下次运行本 notebook 时，已完成的运行会被复用（开启 `USE_DRIVE` 时从 Drive 恢复），只补跑没完成的部分。